In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Convert timestamp columns to timezone-aware datetimes (UTC seconds -> Asia/Shanghai)
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Aggregation helper function
def aggregate_series(df, group_cols, value_col_name):
    # group_cols: list of group columns (e.g., ['cmdb_id','kpi_name'])
    # value_col_name: the name of the value column in df (usually 'value')
    grp = df.groupby(group_cols)[value_col_name]
    agg = grp.agg(
        count='count',
        mean='mean',
        std='std',
        min='min',
        max='max',
        p50=lambda x: x.quantile(0.5),
        p90=lambda x: x.quantile(0.9),
        p95=lambda x: x.quantile(0.95)
    ).reset_index()
    # Sort by count desc
    agg_sorted = agg.sort_values('count', ascending=False).reset_index(drop=True)
    return agg, agg_sorted

# 1) metric.csv aggregation
metric_agg_full, metric_summary = aggregate_series(metric_df, ['cmdb_id','kpi_name'], 'value')
metric_summary = metric_summary[['cmdb_id','kpi_name','count','mean','std','min','max','p50','p90','p95']].head(20)

# 2) trace.csv aggregation
trace_agg_full, trace_summary = aggregate_series(trace_df, ['cmdb_id','trace_name'], 'value')
trace_summary = trace_summary[['cmdb_id','trace_name','count','mean','std','min','max','p50','p90','p95']].head(20)

# 3) log.csv aggregation
log_agg_full, log_summary = aggregate_series(log_df, ['cmdb_id','log_name'], 'value')
log_summary = log_summary[['cmdb_id','log_name','count','mean','std','min','max','p50','p90','p95']].head(20)

# Store full aggregated tables in variables for later steps (already done: metric_agg_full, trace_agg_full, log_agg_full)

# 4) error_logs.csv summary: counts per cmdb_id (top 20), earliest, latest, up to 5 sample messages
if 'message' in error_logs_df.columns:
    samples = error_logs_df.groupby('cmdb_id').agg(
        count=('message','count'),
        earliest=('timestamp','min'),
        latest=('timestamp','max'),
        sample_messages=('message', lambda msgs: list(msgs.head(5)))
    ).reset_index()
    # Convert earliest/latest to strings for compact display
    samples['earliest'] = samples['earliest'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    samples['latest'] = samples['latest'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # Convert sample_messages lists to compact joined string (up to 5)
    samples['sample_messages'] = samples['sample_messages'].apply(lambda lst: ' || '.join(map(str, lst)) )
    error_summary = samples.sort_values('count', ascending=False).head(20)[['cmdb_id','count','earliest','latest','sample_messages']]
else:
    # If no message column, still provide counts and timestamps
    samples = error_logs_df.groupby('cmdb_id').agg(
        count=('timestamp','count'),
        earliest=('timestamp','min'),
        latest=('timestamp','max')
    ).reset_index()
    samples['earliest'] = samples['earliest'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    samples['latest'] = samples['latest'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    error_summary = samples.sort_values('count', ascending=False).head(20)

# Final compact summaries to display (top 20 each)
metric_summary, trace_summary, log_summary, error_summary

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry aggregations (top series):\n\n"
    "metric.csv (top by sample count): Many series have 31 samples. Notable top series:\n"
    "- IG01 : JVM-Threads_7778_JVM_ThreadCount_Threads (count=31, mean≈69.71, std≈2.52)\n"
    "- Mysql02 : several OSLinux CPU metrics (CPUUserTime mean≈1.185, CPUWio mean≈6.63, CPULoad mean≈0.559; all count=31)\n"
    "- Mysql01 / Redis01 also appear in top-20 with uptime/CPU-related metrics (count=31).\n\n"
    "trace.csv (top by sample count): Most top trace series have 30 samples. Notable top series:\n"
    "- IG01 : trace.self.duration_mean (count=30, mean≈0.210s, p95≈0.415s)\n"
    "- IG01 : trace.self.duration_p95 (count=30, mean≈1.162s)\n"
    "- IG01 / IG02 : several to_TomcatX duration_mean and row_count series (counts=30, row_count means ~70–569), indicating consistent tracing across requests.\n\n"
    "log.csv (top by sample count): Many top log series have 31 samples. Notable top series:\n"
    "- Tomcat01 : log.row_count (count=31, mean≈134.5)\n"
    "- apache01 / apache02 : log.row_count also high (means ~267–274, count=31)\n"
    "- log.error_count for multiple Tomcat/apache entries is 0 across the top series (count=31, mean=0).\n\n"
    "error_logs.csv: No aggregated error-log rows returned (empty summary); there are no error_log entries present in the dataset to summarize.\n\n"
    "Overall: The top metric series are mostly OS and JVM CPU/memory/thread metrics (IG01, Mysql02, Redis01). Traces show IG01/IG02 request durations and row counts with p95 values around 0.4–1.2s. Logs show consistent row counts per web/app server and no recorded error_count in the top series. No error_log messages were available for further analysis."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name  count          mean         std           min           max           p50           p90           p95
0      IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  6.970968e+01    2.519174  6.700000e+01  7.500000e+01  6.900000e+01  7.300000e+01  7.450000e+01
1   Mysql02                        OSLinux-CPU_CPU_CPUUserTime     31  1.185071e+00    0.212997  6.937520e-01  1.653600e+00  1.159890e+00  1.454800e+00  1.524515e+00
2   Mysql02                             OSLinux-CPU_CPU_CPUWio     31  6.629801e+00    1.236642  3.921569e+00  9.219300e+00  6.608500e+00  8.364200e+00  8.446700e+00
3   Mysql02                        OSLinux-CPU_CPU_CPUidleutil     31  9.102783e+01    1.602089  8.762310e+01  9.459538e+01  9.096997e+01  9.260398e+01  9.345227e+01
4   Mysql02                            OSLinux-CPU_CPU_CPULoad     31  5.587097e-01    0.208082  1.400000e-01  9.900000e-01  5.300000e-01  8.300000e-01  8.900000e-01
5   Mysql01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.274686e+06  547.478077  6.273782e+06  6.275588e+06  6.274685e+06  6.275408e+06  6.275498e+06
6   Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00    0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
7   Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00    0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
8   Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00    0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
9   Mysql02                         OSLinux-CPU_CPU_CPUSysTime     31  9.273424e-01    0.131543  6.688270e-01  1.197000e+00  8.977560e-01  1.113500e+00  1.126678e+00
10  Mysql02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.274731e+06  547.126073  6.273828e+06  6.275634e+06  6.274732e+06  6.275454e+06  6.275543e+06
11  Redis01                            OSLinux-CPU_CPU_CPULoad     31  1.037419e+00    0.039914  1.000000e+00  1.130000e+00  1.020000e+00  1.100000e+00  1.125000e+00
12  Mysql02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  5.129032e-01    0.034078  5.000000e-01  6.000000e-01  5.000000e-01  6.000000e-01  6.000000e-01
13  Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  1.600000e+01    0.000000  1.600000e+01  1.600000e+01  1.600000e+01  1.600000e+01  1.600000e+01
14  Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00    0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
15  Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00    0.000000  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
16  Redis01                         OSLinux-CPU_CPU_CPUSysTime     31  9.045310e-01    0.151935  6.401460e-01  1.293677e+00  8.939000e-01  1.056836e+00  1.225195e+00
17  Redis01                        OSLinux-CPU_CPU_CPUUserTime     31  2.518105e+01    0.052630  2.506240e+01  2.528286e+01  2.517363e+01  2.524120e+01  2.526450e+01
18  Redis01                             OSLinux-CPU_CPU_CPUWio     31  6.104097e-02    0.131201  4.157000e-03  5.200000e-01  8.314000e-03  1.829900e-01  3.618000e-01
19  Redis01                        OSLinux-CPU_CPU_CPUidleutil     31  7.327091e+01    0.305955  7.237100e+01  7.382467e+01  7.330699e+01  7.358950e+01  7.369185e+01,    cmdb_id                       trace_name  count        mean         std         min         max         p50         p90         p95
0     IG01         trace.self.duration_mean     30    0.210195    0.111220    0.080177    0.531085    0.168948    0.342634    0.415401
1     IG01          trace.self.duration_p95     30    1.161575    0.470919    0.333000    2.778000    1.059150    1.533505    2.066835
2     IG01             trace.self.row_count     30  568.600000  121.170328  282.000000  802.000000  563.000000  724.600000  735.500000
3     IG01  trace.to_Tomcat01.duration_mean     30    0.170391    0.074599    0.093528    0.388432    0.144912    0.252254    0.309493
4     IG01   trace.to_Tomcat01.duration_p95     30    0.948550    0.232038    0.494500    1.447500    1.032800    1.126540    1.155140
5     IG01      trace.to_Tomcat01.row_count     30   71.400000   18.605895   33.000000  106.000000   71.000000   92.500000   98.100000
6     IG01  trace.to_Tomcat02.duration_mean     30    0.188001    0.104945    0.058250    0.563188    0.170356    0.244581    0.384953
7     IG01   trace.to_Tomcat02.duration_p95     30    1.025990    0.394826    0.148350    2.187000    1.050250    1.155850    1.696437
8     IG01      trace.to_Tomcat02.row_count     30   70.866667   17.621956   34.000000  103.000000   69.500000   90.600000   99.300000
9     IG01  trace.to_Tomcat03.duration_mean     30    0.302157    0.331497    0.052714    1.637902    0.184391    0.781539    0.798152
10    IG01   trace.to_Tomcat03.duration_p95     30    1.202802    0.720283    0.074750    3.659000    1.054250    2.114230    2.382150
11    IG01      trace.to_Tomcat03.row_count     30   70.066667   15.682042   35.000000  116.000000   70.500000   83.100000   88.950000
12    IG01  trace.to_Tomcat04.duration_mean     30    0.168198    0.098099    0.048172    0.499080    0.150598    0.251523    0.379402
13    IG01   trace.to_Tomcat04.duration_p95     30    0.949377    0.440028    0.141300    2.127300    1.030275    1.162125    1.680255
14    IG01      trace.to_Tomcat04.row_count     30   71.966667   15.533020   38.000000  100.000000   73.500000   90.300000   94.650000
15    IG02         trace.self.duration_mean     30    0.224081    0.113859    0.083638    0.534593    0.185318    0.403534    0.443058
16    IG02          trace.self.duration_p95     30    1.219953    0.486994    0.429600    2.913000    1.060000    1.785445    2.110002
17    IG02             trace.self.row_count     30  548.466667  116.038856  304.000000  822.000000  540.000000  690.800000  739.800000
18    IG02  trace.to_Tomcat01.duration_mean     30    0.179818    0.081683    0.052190    0.433341    0.168413    0.284792    0.304172
19    IG02   trace.to_Tomcat01.duration_p95     30    1.001093    0.380772    0.144550    2.104600    1.055300    1.243935    1.517330,      cmdb_id          log_name  count        mean        std   min     max      p50      p90       p95
0   Tomcat01     log.row_count     31  134.516129  39.443522  4.00  205.00  139.000  179.000  189.5000
1   Tomcat01   log.error_count     31    0.000000   0.000000  0.00    0.00    0.000    0.000    0.0000
2   apache02   log.error_count     31    0.000000   0.000000  0.00    0.00    0.000    0.000    0.0000
3   apache01     log.row_count     31  267.225806  74.663561  7.00  407.00  267.000  338.000  358.5000
4   Tomcat02   log.error_count     31    0.000000   0.000000  0.00    0.00    0.000    0.000    0.0000
5   Tomcat02     log.row_count     31  136.741935  40.301338  5.00  220.00  138.000  172.000  196.0000
6   Tomcat03   log.error_count     31    0.000000   0.000000  0.00    0.00    0.000    0.000    0.0000
7   Tomcat04     log.row_count     31  135.645161  37.627604  2.00  182.00  141.000  175.000  180.5000
8   Tomcat04   log.error_count     31    0.000000   0.000000  0.00    0.00    0.000    0.000    0.0000
9   Tomcat03     log.row_count     31  135.419355  38.862814  3.00  216.00  139.000  172.000  180.5000
10  apache02     log.row_count     31  273.709677  76.729913  6.00  405.00  274.000  361.000  382.0000
11  apache01   log.error_count     31    0.000000   0.000000  0.00    0.00    0.000    0.000    0.0000
12  Tomcat02  log.gc_pause_max     20    0.067500   0.020743  0.02    0.09    0.075    0.090    0.0900
13   Mysql01   log.error_count     12    0.000000   0.000000  0.00    0.00    0.000    0.000    0.0000
14   Mysql01     log.row_count     12    1.750000   0.866025  1.00    3.00    1.500    3.000    3.0000
15  Tomcat01  log.gc_pause_max      8    0.056250   0.026152  0.02    0.09    0.055    0.090    0.0900
16  Tomcat03  log.gc_pause_max      8    0.060000   0.023905  0.02    0.09    0.065    0.083    0.0865
17  Tomcat04  log.gc_pause_max      8    0.066250   0.022638  0.02    0.09    0.075    0.083    0.0865, Empty DataFrame
Columns: [cmdb_id, count, earliest, latest, sample_messages]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing dataframes and tz if available; otherwise load
try:
    metric_df
    trace_df
    log_df
    error_logs_df
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')
    metric_df = pd.read_csv('metric.csv')
    trace_df = pd.read_csv('trace.csv')
    log_df = pd.read_csv('log.csv')
    error_logs_df = pd.read_csv('error_logs.csv')
    for df in (metric_df, trace_df, log_df, error_logs_df):
        if 'timestamp' in df.columns:
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure timestamps are timezone-aware Asia/Shanghai (UTC+8)
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns and df['timestamp'].dt.tz is None:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global thresholds (p95 and p05) per aggregated series using full series (no filtering)
metric_thresholds = metric_df.groupby(['cmdb_id','kpi_name'])['value'].agg(
    global_p95=lambda x: x.quantile(0.95),
    global_p05=lambda x: x.quantile(0.05)
).reset_index()

trace_thresholds = trace_df.groupby(['cmdb_id','trace_name'])['value'].agg(
    global_p95=lambda x: x.quantile(0.95),
    global_p05=lambda x: x.quantile(0.05)
).reset_index()

log_thresholds = log_df.groupby(['cmdb_id','log_name'])['value'].agg(
    global_p95=lambda x: x.quantile(0.95),
    global_p05=lambda x: x.quantile(0.05)
).reset_index()

# Store full aggregated threshold tables for later use (kernel variables)
metric_thresholds_full = metric_thresholds.copy()
trace_thresholds_full = trace_thresholds.copy()
log_thresholds_full = log_thresholds.copy()

# 2) Define anomaly rules helper
latency_resource_keywords = ['duration','cpu','memory','heap','threads','load','wio','systime','usertime','gc_pause','oom']
throughput_keywords = ['row_count','rows','throughput']
# Special log error name
error_name_exact = 'log.error_count'

def classify_kpi(name):
    ln = str(name).lower()
    # log.error_count special-case
    if ln == error_name_exact:
        return 'error_count'
    # throughput-like
    for kw in throughput_keywords:
        if kw in ln:
            return 'low_is_bad'  # throughput/traffic: low is bad
    # latency/resource-like
    for kw in latency_resource_keywords:
        if kw in ln:
            return 'high_is_bad'
    # default: higher-is-bad
    return 'high_is_bad'

# 3) Filter to incident window (UTC+8): March 7, 2021 08:00:00 to 08:30:00 inclusive
start = pd.Timestamp('2021-03-07 08:00:00', tz=tz)
end = pd.Timestamp('2021-03-07 08:30:00', tz=tz)

metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()
trace_window = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()
log_window = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)].copy()
error_window = error_logs_df[(error_logs_df['timestamp'] >= start) & (error_logs_df['timestamp'] <= end)].copy()

# Helper to apply thresholds and flag anomalies per row
def apply_anomaly_rules(df_window, thresholds_df, group_cols, name_col):
    # Merge thresholds onto windowed records
    merged = df_window.merge(thresholds_df, on=group_cols, how='left')
    # classify per-row based on name_col
    types = merged[name_col].apply(classify_kpi)
    merged['kpi_type'] = types
    # Determine anomaly boolean
    def is_anom(row):
        val = row['value']
        typ = row['kpi_type']
        p95 = row['global_p95']
        p05 = row['global_p05']
        if typ == 'error_count':
            return bool(val > 0)
        if typ == 'high_is_bad':
            # If p95 is NaN (no threshold), be conservative: treat as False
            if pd.isna(p95):
                return False
            return val >= p95
        if typ == 'low_is_bad':
            if pd.isna(p05):
                return False
            return val <= p05
        # default
        if pd.isna(p95):
            return False
        return val >= p95

    merged['is_anomaly'] = merged.apply(is_anom, axis=1)
    return merged

metric_merged = apply_anomaly_rules(metric_window, metric_thresholds, ['cmdb_id','kpi_name'], 'kpi_name')
trace_merged = apply_anomaly_rules(trace_window, trace_thresholds, ['cmdb_id','trace_name'], 'trace_name')
log_merged = apply_anomaly_rules(log_window, log_thresholds, ['cmdb_id','log_name'], 'log_name')

# Extract anomaly rows
metric_anoms = metric_merged[metric_merged['is_anomaly']].copy()
trace_anoms = trace_merged[trace_merged['is_anomaly']].copy()
log_anoms = log_merged[log_merged['is_anomaly']].copy()

# 3) For each file produce compact aggregated anomaly summary
def build_anomaly_summary(anom_df, group_cols, name_col):
    if anom_df.empty:
        cols = ['component','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value','global_p95','global_p05','severity_ratio','sample_anomalies']
        return pd.DataFrame(columns=cols)
    # Prepare sample tuples as formatted strings
    anom_df['ts_str'] = anom_df['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    anom_df['sample_tuple'] = anom_df.apply(lambda r: f"({r['ts_str']}, {r['value']})", axis=1)
    # Group and aggregate
    grp = anom_df.groupby(group_cols).agg(
        anomaly_count=('is_anomaly','size'),
        earliest_anomaly_timestamp=('timestamp','min'),
        max_value=('value','max'),
        min_value=('value','min'),
        global_p95=('global_p95','first'),
        global_p05=('global_p05','first'),
        sample_list = ('sample_tuple', lambda s: ' || '.join(s.iloc[:5].tolist()))
    ).reset_index()
    # Determine KPI type for severity ratio (use the name column)
    grp['kpi_type'] = grp[name_col].apply(classify_kpi)
    # Compute severity_ratio
    def compute_sev(row):
        typ = row['kpi_type']
        p95 = row['global_p95']
        p05 = row['global_p05']
        maxv = row['max_value']
        minv = row['min_value']
        if typ == 'low_is_bad':
            # severity = global_p05 / min_value (higher worse)
            if (minv == 0) or pd.isna(p05):
                return np.nan
            return float(p05) / float(minv)
        else:
            # high-is-bad
            if (pd.isna(p95)) or (p95 == 0):
                return np.nan
            return float(maxv) / float(p95)
    grp['severity_ratio'] = grp.apply(compute_sev, axis=1)
    # Format earliest timestamp
    grp['earliest_anomaly_timestamp'] = grp['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # Rename columns to required output names
    out = grp.rename(columns={
        group_cols[0]: 'component',
        name_col: 'kpi_name'
    })[['component','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value','global_p95','global_p05','severity_ratio','sample_list']]
    out = out.sort_values('anomaly_count', ascending=False).reset_index(drop=True)
    out = out.rename(columns={'sample_list':'sample_anomalies'})
    return out

metric_anom_summary = build_anomaly_summary(metric_anoms, ['cmdb_id','kpi_name'], 'kpi_name').head(20)
trace_anom_summary = build_anomaly_summary(trace_anoms, ['cmdb_id','trace_name'], 'trace_name').head(20)
log_anom_summary = build_anomaly_summary(log_anoms, ['cmdb_id','log_name'], 'log_name').head(20)

# Overall counts and components with anomalies (top 20)
def overall_component_counts(anom_df):
    total_anoms = int(anom_df.shape[0])
    if anom_df.empty:
        comp_df = pd.DataFrame(columns=['component','anomaly_count'])
    else:
        comp_df = anom_df.groupby('cmdb_id').size().reset_index(name='anomaly_count').rename(columns={'cmdb_id':'component'}).sort_values('anomaly_count', ascending=False).head(20)
    return total_anoms, comp_df

metric_total_anoms, metric_components_with_anoms = overall_component_counts(metric_anoms)
trace_total_anoms, trace_components_with_anoms = overall_component_counts(trace_anoms)
log_total_anoms, log_components_with_anoms = overall_component_counts(log_anoms)

# 4) error_logs within window summary
if error_window.empty:
    # explicit empty summary
    error_window_summary = pd.DataFrame(columns=['cmdb_id','count','earliest','latest','sample_messages'])
else:
    # aggregate per cmdb_id
    error_window['ts_str'] = error_window['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    error_samples = error_window.groupby('cmdb_id').agg(
        count=('message','count'),
        earliest=('timestamp','min'),
        latest=('timestamp','max'),
        sample_messages=('message', lambda s: ' || '.join(s.astype(str).head(5).tolist()))
    ).reset_index()
    error_samples['earliest'] = error_samples['earliest'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    error_samples['latest'] = error_samples['latest'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    error_window_summary = error_samples.sort_values('count', ascending=False).head(20)

# Final compact outputs (only these variables will be displayed)
metric_anom_summary, metric_total_anoms, metric_components_with_anoms, \
trace_anom_summary, trace_total_anoms, trace_components_with_anoms, \
log_anom_summary, log_total_anoms, log_components_with_anoms, \
error_window_summary

```
Out[2]:
```
Summary of anomaly detection for the incident window (2021-03-07 08:00:00 — 08:30:00, UTC+8). Global thresholds (p95, p05) were computed from the full series before window filtering.

High-level totals
- Metrics: 15,813 anomalous points in-window.
- Traces: 484 anomalous points in-window.
- Logs: 27 anomalous points in-window.
- error_logs: no records in the incident window (empty).

Notes about thresholds and many alerts
- Many metric series have global p95 == 0 (and p05 == 0). Because the anomaly rule marks value >= p95 as anomalous for “high-is-bad” KPIs, zero-valued series produced many anomaly flags (value 0 >= p95 0). This inflates metric anomaly counts and yields NaN severity_ratio for those series (division by zero / undefined).
- Where global thresholds are non-zero, severity_ratio is shown as max_value / global_p95 (for high-is-bad) or global_p05 / min_value (for low-is-bad throughput).

Top components by anomaly counts (metrics)
- Top components with metric anomaly counts (top sample): 
  1) Mysql01 — 2,387 anomalous points
  2) Mysql02 — 2,040
  3) Redis01 — 1,435
  4) Redis02 — 1,415
  5) Tomcat04 — 1,050
  6) Tomcat02 — 1,022
  7) Tomcat03 — 1,002
  8) Tomcat01 — 973
  9) IG02 — 671
 10) IG01 — 670
 11) apache02 — 629
 12) MG02 — 614
(remaining components shown in full table)

Representative metric anomalies (top rows)
- Many OSLinux / process / session metrics are flagged with anomaly_count = 31 (the full 31-minute window), often because values equal the global p95 (or p95==0).
- Example flagged metrics for Tomcat01: ErrorCount=32, MaxTimeRequests=348,847, MaxThreads=500 (these values equal their global p95 and produce severity_ratio = 1.0).
- Several Redis and Apache OS/process counters appear repeatedly with zero values where p95==0, producing many flagged points (likely noisy/false positives).

Traces: summary and notable anomalies
- Total trace anomalies: 484.
- Top components by trace anomalies (sample): MG02 (81), MG01 (78), dockerB2 (37), dockerA2 (36), dockerB1 (33), dockerA1 (33), Tomcat03 (33), IG01 (31), Tomcat02 (31), Tomcat01 (31), IG02 (30), Tomcat04 (30).
- Representative trace anomalies:
  - dockerB2.trace.self.duration_p95: 9 anomalies (max 0.01, p95 0.01, severity_ratio 1.0).
  - dockerA2.trace.self.duration_p95: 6 anomalies (max 0.01885, global_p95 0.011 → severity_ratio ≈ 1.71).
  - Several row_count (throughput) series flagged as low-is-bad (value <= global p05), e.g., to_MG02.row_count observed values of 5 (much lower than typical p05/p95), with severity_ratio computed accordingly.
  - IG01/IG02 trace durations and row_counts: some p95-exceeding durations and some row_count drops (severity ratios > 1 for duration anomalies).

Logs: summary and notable anomalies
- Total log anomalies: 27.
- Top components by log anomalies (sample): Tomcat02 (7), Mysql01 (6), Tomcat01 (4), Tomcat03 (3), Tomcat04 (3), apache01 (2), apache02 (2).
- Representative log anomalies:
  - Mysql01.log.row_count: 6 anomalies where row_count dropped to 1 (global p05 = 1), severity_ratio shows p05/min_value = 1.0.
  - Tomcat02 log.gc_pause_max: 5 anomalies hitting 0.09 (global_p95 = 0.09, severity_ratio 1.0).
  - Several servers show isolated low row_count events (throughput anomalies) and occasional gc_pause spikes.

error_logs
- No error log entries found in the incident window (explicit empty summary).

Practical interpretation and next steps
- The raw counts are dominated by many zero-valued series whose global p95==0 — these produce many flagged anomalies but are likely noisy or false positives. Investigate whether those KPIs are meaningful (are zero values valid?) before acting.
- Focus first on anomalies with:
  - non-zero thresholds and severity_ratio > 1.0 (actual value exceeds global p95), e.g., dockerA2 trace duration p95 (severity ≈1.71), IG01/IG02 duration p95s above normal, Tomcat Request MaxTime and MaxThreads at their historic maxima.
  - throughput (row_count) drops that are significant (severity_ratio >>1 for p05/min_value) — these indicate reduced traffic or dropped processing in certain components (several IG/Tomcat/MG rows shown).
- Logs show few anomalies; check Mysql01 and Tomcat gc_pause / row_count anomalies for correlation with trace latency spikes.
- Because error_logs are empty in-window, there are no explicit application error messages to correlate.

If you want, I can:
- produce a filtered list of the highest-severity anomalies (excluding p95==0 cases),
- or show the raw anomaly samples for a particular component (e.g., Mysql01 or IG01) to drill into timestamps and values.

The original code execution output of IPython Kernel is also provided below for reference:

(   component                                           kpi_name  anomaly_count earliest_anomaly_timestamp  max_value  global_p95  global_p05  severity_ratio                                   sample_anomalies
0   apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
1   apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
2   Tomcat01     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
3   Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
4   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31    2021-03-07 08:00:00 CST       32.0        32.0        32.0             1.0  (2021-03-07 08:00:00 CST, 32.0) || (2021-03-07...
5   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31    2021-03-07 08:00:00 CST   348847.0    348847.0    348847.0             1.0  (2021-03-07 08:00:00 CST, 348847.0) || (2021-0...
6   Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31    2021-03-07 08:00:00 CST      500.0       500.0       500.0             1.0  (2021-03-07 08:00:00 CST, 500.0) || (2021-03-0...
7   Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
8   Tomcat01    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
9   Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
10  Tomcat01        Tomcat-Sessions_7441--_SESSIONActiveCounter             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
11   Redis01          redis-Redis_6379_Redis  (pubsub_channels)             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
12  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
13  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
14  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
15   Redis02   redis-Redis_6379_Redis  (rdb_bgsave_in_progress)             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
16   Redis02          redis-Redis_6379_Redis  (redis_git_dirty)             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
17   Redis02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ...
18   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31    2021-03-07 08:00:00 CST        4.0         4.0         4.0             1.0  (2021-03-07 08:00:00 CST, 4.0) || (2021-03-07 ...
19   Redis02    redis-Redis_6379_Redis  (aof_rewrite_scheduled)             31    2021-03-07 08:00:00 CST        0.0         0.0         0.0             NaN  (2021-03-07 08:00:00 CST, 0.0) || (2021-03-07 ..., 15813,         component  anomaly_count
4         Mysql01           2387
5         Mysql02           2040
6         Redis01           1435
7         Redis02           1415
22       Tomcat04           1050
20       Tomcat02           1022
21       Tomcat03           1002
19       Tomcat01            973
1            IG02            671
0            IG01            670
24       apache02            629
3            MG02            614
2            MG01            595
23       apache01            539
12   ServiceTest3             66
10  ServiceTest11             66
9   ServiceTest10             66
8    ServiceTest1             66
11   ServiceTest2             66
13   ServiceTest4             66,    component                         kpi_name  anomaly_count earliest_anomaly_timestamp   max_value  global_p95  global_p05  severity_ratio                                   sample_anomalies
0   dockerB2          trace.self.duration_p95              9    2021-03-07 08:03:00 CST    0.010000    0.010000    0.009000        1.000000  (2021-03-07 08:03:00 CST, 0.01) || (2021-03-07...
1   dockerA2          trace.self.duration_p95              6    2021-03-07 08:11:00 CST    0.018850    0.011000    0.003000        1.713636  (2021-03-07 08:11:00 CST, 0.011) || (2021-03-0...
2   dockerB1          trace.self.duration_p95              5    2021-03-07 08:17:00 CST    0.012000    0.012000    0.010000        1.000000  (2021-03-07 08:17:00 CST, 0.012) || (2021-03-0...
3   dockerA2          trace.to_MG02.row_count              4    2021-03-07 08:02:00 CST    5.000000   25.100000    5.000000        2.500000  (2021-03-07 08:02:00 CST, 5.0) || (2021-03-07 ...
4       MG02    trace.from_dockerA2.row_count              4    2021-03-07 08:02:00 CST    5.000000   25.100000    5.000000        2.500000  (2021-03-07 08:02:00 CST, 5.0) || (2021-03-07 ...
5   Tomcat01        trace.from_IG01.row_count              3    2021-03-07 08:00:00 CST   36.000000   98.100000   36.000000        1.090909  (2021-03-07 08:00:00 CST, 33.0) || (2021-03-07...
6       MG02    trace.from_Tomcat02.row_count              3    2021-03-07 08:00:00 CST   42.000000  101.550000   42.000000        1.200000  (2021-03-07 08:00:00 CST, 42.0) || (2021-03-07...
7       IG01      trace.to_Tomcat01.row_count              3    2021-03-07 08:00:00 CST   36.000000   98.100000   36.000000        1.090909  (2021-03-07 08:00:00 CST, 33.0) || (2021-03-07...
8   Tomcat02          trace.to_MG02.row_count              3    2021-03-07 08:00:00 CST   42.000000  101.550000   42.000000        1.200000  (2021-03-07 08:00:00 CST, 42.0) || (2021-03-07...
9       IG01             trace.self.row_count              2    2021-03-07 08:00:00 CST  300.000000  735.500000  327.900000        1.162766  (2021-03-07 08:00:00 CST, 282.0) || (2021-03-0...
10      IG01   trace.to_Tomcat03.duration_p95              2    2021-03-07 08:12:00 CST    3.659000    2.382150    0.270927        1.536007  (2021-03-07 08:12:00 CST, 2.42625) || (2021-03...
11      IG01      trace.to_Tomcat03.row_count              2    2021-03-07 08:00:00 CST   36.000000   88.950000   44.100000        1.260000  (2021-03-07 08:00:00 CST, 36.0) || (2021-03-07...
12      IG01  trace.to_Tomcat04.duration_mean              2    2021-03-07 08:23:00 CST    0.499080    0.379402    0.073757        1.315440  (2021-03-07 08:23:00 CST, 0.466446) || (2021-0...
13      IG01   trace.to_Tomcat04.duration_p95              2    2021-03-07 08:23:00 CST    2.127300    1.680255    0.244062        1.266058  (2021-03-07 08:23:00 CST, 2.0889) || (2021-03-...
14      IG01      trace.to_Tomcat04.row_count              2    2021-03-07 08:00:00 CST   43.000000   94.650000   43.900000        1.155263  (2021-03-07 08:00:00 CST, 38.0) || (2021-03-07...
15      IG02         trace.self.duration_mean              2    2021-03-07 08:15:00 CST    0.534593    0.443058    0.101570        1.206598  (2021-03-07 08:15:00 CST, 0.534593) || (2021-0...
16      IG02          trace.self.duration_p95              2    2021-03-07 08:15:00 CST    2.913000    2.110002    0.576300        1.380567  (2021-03-07 08:15:00 CST, 2.913) || (2021-03-0...
17      IG02             trace.self.row_count              2    2021-03-07 08:00:00 CST  310.000000  739.800000  330.700000        1.087829  (2021-03-07 08:00:00 CST, 310.0) || (2021-03-0...
18      IG02  trace.to_Tomcat01.duration_mean              2    2021-03-07 08:23:00 CST    0.433341    0.304172    0.076972        1.424658  (2021-03-07 08:23:00 CST, 0.433341) || (2021-0...
19      IG02   trace.to_Tomcat01.duration_p95              2    2021-03-07 08:09:00 CST    2.104600    1.517330    0.344135        1.387042  (2021-03-07 08:09:00 CST, 1.73855) || (2021-03..., 484,    component  anomaly_count
3       MG02             81
2       MG01             78
11  dockerB2             37
9   dockerA2             36
10  dockerB1             33
8   dockerA1             33
6   Tomcat03             33
0       IG01             31
5   Tomcat02             31
4   Tomcat01             31
1       IG02             30
7   Tomcat04             30,    component          kpi_name  anomaly_count earliest_anomaly_timestamp  max_value  global_p95  global_p05  severity_ratio                                   sample_anomalies
0    Mysql01     log.row_count              6    2021-03-07 08:01:00 CST       1.00      3.0000      1.0000        1.000000  (2021-03-07 08:01:00 CST, 1.0) || (2021-03-07 ...
1   Tomcat02  log.gc_pause_max              5    2021-03-07 08:05:00 CST       0.09      0.0900      0.0390        1.000000  (2021-03-07 08:05:00 CST, 0.09) || (2021-03-07...
2   Tomcat01  log.gc_pause_max              2    2021-03-07 08:02:00 CST       0.09      0.0900      0.0235        1.000000  (2021-03-07 08:02:00 CST, 0.09) || (2021-03-07...
3   Tomcat01     log.row_count              2    2021-03-07 08:00:00 CST      75.00    189.5000     76.5000       19.125000  (2021-03-07 08:00:00 CST, 75.0) || (2021-03-07...
4   Tomcat02     log.row_count              2    2021-03-07 08:01:00 CST      73.00    196.0000     75.0000       15.000000  (2021-03-07 08:01:00 CST, 73.0) || (2021-03-07...
5   Tomcat03     log.row_count              2    2021-03-07 08:01:00 CST      67.00    180.5000     69.0000       23.000000  (2021-03-07 08:01:00 CST, 67.0) || (2021-03-07...
6   Tomcat04     log.row_count              2    2021-03-07 08:00:00 CST      75.00    180.5000     79.0000       39.500000  (2021-03-07 08:00:00 CST, 75.0) || (2021-03-07...
7   apache02     log.row_count              2    2021-03-07 08:00:00 CST     152.00    382.0000    153.0000       25.500000  (2021-03-07 08:00:00 CST, 152.0) || (2021-03-0...
8   apache01     log.row_count              2    2021-03-07 08:00:00 CST     144.00    358.5000    146.0000       20.857143  (2021-03-07 08:00:00 CST, 144.0) || (2021-03-0...
9   Tomcat03  log.gc_pause_max              1    2021-03-07 08:13:00 CST       0.09      0.0865      0.0235        1.040462                    (2021-03-07 08:13:00 CST, 0.09)
10  Tomcat04  log.gc_pause_max              1    2021-03-07 08:26:00 CST       0.09      0.0865      0.0305        1.040462                    (2021-03-07 08:26:00 CST, 0.09), 27,   component  anomaly_count
2  Tomcat02              7
0   Mysql01              6
1  Tomcat01              4
3  Tomcat03              3
4  Tomcat04              3
5  apache01              2
6  apache02              2, Empty DataFrame
Columns: [cmdb_id, count, earliest, latest, sample_messages]
Index: [])```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables where available; otherwise load CSVs and prepare
tz = pytz.timezone('Asia/Shanghai')

try:
    metric_df, metric_thresholds_full, metric_merged, metric_window
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # compute global thresholds
    metric_thresholds_full = metric_df.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        global_p05=lambda x: x.quantile(0.05)
    ).reset_index()
    # incident window
    start = pd.Timestamp('2021-03-07 08:00:00', tz=tz)
    end = pd.Timestamp('2021-03-07 08:30:00', tz=tz)
    metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()
    # classification helper
    latency_resource_keywords = ['duration','duration_p95','cpu','memory','heap','threads','load','wio','systime','usertime','gc_pause','oom','maxtime','maxthreads']
    throughput_keywords = ['row_count','rows','throughput']
    def classify_kpi(name):
        ln = str(name).lower()
        if 'log.error_count' in ln:
            return 'error_count'
        for kw in throughput_keywords:
            if kw in ln:
                return 'low_is_bad'
        for kw in latency_resource_keywords:
            if kw in ln:
                return 'high_is_bad'
        return 'high_is_bad'
    # merge thresholds and set anomaly flag as previous logic
    metric_merged = metric_window.merge(metric_thresholds_full, on=['cmdb_id','kpi_name'], how='left')
    metric_merged['kpi_type'] = metric_merged['kpi_name'].apply(classify_kpi)
    def is_anom_row(r):
        val = r['value']
        typ = r['kpi_type']
        p95 = r['global_p95']
        p05 = r['global_p05']
        if typ == 'error_count':
            return bool(val > 0)
        if typ == 'low_is_bad':
            if pd.isna(p05): return False
            return val <= p05
        # high_is_bad default
        if pd.isna(p95): return False
        return val >= p95
    metric_merged['is_anomaly'] = metric_merged.apply(is_anom_row, axis=1)

# Ensure thresholds exist (recompute if needed)
if 'metric_thresholds_full' not in globals():
    metric_thresholds_full = metric_df.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        global_p05=lambda x: x.quantile(0.05)
    ).reset_index()

# Define keywords for stricter selection (per user)
high_keywords = ['duration','duration_p95','cpu','memory','heap','threads','load','wio','systime','usertime','gc_pause','oom','maxtime','maxthreads']
low_keywords = ['row_count','rows','throughput']

def is_high_kpi(name):
    ln = str(name).lower()
    return any(kw in ln for kw in high_keywords)

def is_low_kpi(name):
    ln = str(name).lower()
    return any(kw in ln for kw in low_keywords)

# Work on metric_merged: keep only rows where is_anomaly True (these are in-window anomaly points)
metric_anomaly_rows = metric_merged[metric_merged['is_anomaly']].copy()
# sort for grouping
metric_anomaly_rows = metric_anomaly_rows.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

# Convert timestamp to minute resolution (should already be floored)
# We'll use timestamp differences of exactly 1 minute to find consecutive blocks
one_min = pd.Timedelta(minutes=1)

# Build fault intervals per (cmdb_id,kpi_name)
interval_records = []  # will store all intervals that meet initial anomaly contiguous grouping (even if later filtered)
grouped = metric_anomaly_rows.groupby(['cmdb_id','kpi_name'])

for (cmdb_id, kpi_name), grp in grouped:
    grp = grp.sort_values('timestamp').reset_index(drop=True)
    if grp.empty:
        continue
    # compute breaks when difference != 1 minute
    grp['ts_diff'] = grp['timestamp'].diff().fillna(pd.Timedelta(seconds=0))
    # start a new block when ts_diff != 1min
    grp['block_id'] = (grp['ts_diff'] != one_min).cumsum()
    for bid, block in grp.groupby('block_id'):
        start_ts = block['timestamp'].min()
        end_ts = block['timestamp'].max()
        length_minutes = int(((end_ts - start_ts).total_seconds() // 60) + 1)  # inclusive minutes count
        max_val = float(block['value'].max())
        min_val = float(block['value'].min())
        # get global thresholds from metric_thresholds_full
        thr = metric_thresholds_full[(metric_thresholds_full['cmdb_id']==cmdb_id) & (metric_thresholds_full['kpi_name']==kpi_name)]
        if thr.empty:
            global_p95 = np.nan
            global_p05 = np.nan
        else:
            global_p95 = float(thr['global_p95'].iloc[0])
            global_p05 = float(thr['global_p05'].iloc[0])
        # classify KPI type by name
        kpi_low = is_low_kpi(kpi_name)
        kpi_high = is_high_kpi(kpi_name)
        is_error_count = (str(kpi_name).lower() == 'log.error_count')
        # compute severity candidate
        severity = np.nan
        if is_error_count:
            # severity for error_count can be max_val (count) / 1 (no p95) -- but treat as max_val
            severity = float(max_val)
        elif kpi_low:
            if min_val == 0 or pd.isna(global_p05):
                severity = np.nan
            else:
                severity = float(global_p05) / float(min_val) if min_val != 0 else np.nan
        else:
            # default high-is-bad
            if pd.isna(global_p95) or global_p95 == 0:
                severity = np.nan
            else:
                severity = float(max_val) / float(global_p95)
        interval_records.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'interval_start': start_ts,
            'interval_end': end_ts,
            'length_minutes': length_minutes,
            'max_value': max_val,
            'min_value': min_val,
            'global_p95': global_p95,
            'global_p05': global_p05,
            'kpi_is_high': kpi_high,
            'kpi_is_low': kpi_low,
            'is_error_count': is_error_count,
            'severity_ratio': severity
        })

# Create DataFrame of all contiguous anomaly intervals
fault_intervals_df = pd.DataFrame(interval_records)
# Store full intermediate table
metric_fault_intervals_full = fault_intervals_df.copy()

# Now apply stricter selection rules:
# 1) For high-is-bad: keep intervals where global_p95 > 0, max_value >= 1.5*global_p95, length>=2
# 2) For low-is-bad: keep intervals where global_p05 > 0, min_value <= 0.5*global_p05, length>=2
# 3) For log.error_count: include any interval (severity based on max_value), require length>=1? user said treat each consecutive block as fault interval; keep them
selected_intervals = []

for _, row in fault_intervals_df.iterrows():
    if row['is_error_count']:
        # include if any value>0 in interval (by construction anomalies had >0), keep block (no length threshold stated)
        if row['max_value'] > 0:
            selected_intervals.append(row.to_dict())
        continue
    if row['kpi_is_high']:
        if (not pd.isna(row['global_p95'])) and (row['global_p95'] > 0):
            if (row['max_value'] >= 1.5 * row['global_p95']) and (row['length_minutes'] >= 2):
                selected_intervals.append(row.to_dict())
        # else skip (noise or zero threshold)
        continue
    if row['kpi_is_low']:
        if (not pd.isna(row['global_p05'])) and (row['global_p05'] > 0):
            if (row['min_value'] <= 0.5 * row['global_p05']) and (row['length_minutes'] >= 2):
                selected_intervals.append(row.to_dict())
        continue
    # default high-is-bad behavior
    if (not pd.isna(row['global_p95'])) and (row['global_p95'] > 0):
        if (row['max_value'] >= 1.5 * row['global_p95']) and (row['length_minutes'] >= 2):
            selected_intervals.append(row.to_dict())

selected_intervals_df = pd.DataFrame(selected_intervals)

# Aggregate per (cmdb_id,kpi_name): number_of_fault_intervals and up to 5 intervals info
if selected_intervals_df.empty:
    metric_fault_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','number_of_fault_intervals','intervals_summary'
    ])
    candidate_components_count = 0
else:
    # prepare formatting of interval tuples
    selected_intervals_df['interval_start_str'] = selected_intervals_df['interval_start'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    selected_intervals_df['interval_end_str'] = selected_intervals_df['interval_end'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    selected_intervals_df = selected_intervals_df.sort_values(['severity_ratio'], ascending=False)
    grouped_sel = selected_intervals_df.groupby(['cmdb_id','kpi_name'])
    summary_rows = []
    for (cmdb_id,kpi_name), g in grouped_sel:
        g = g.sort_values('severity_ratio', ascending=False)
        num_intervals = g.shape[0]
        # build up to 5 interval summaries as tuple strings
        interval_strs = []
        for _, r in g.head(5).iterrows():
            tup = (
                r['interval_start_str'],
                r['interval_end_str'],
                int(r['length_minutes']),
                float(r['max_value']),
                float(r['min_value']),
                float(r['severity_ratio']) if not pd.isna(r['severity_ratio']) else None
            )
            # format compactly
            interval_strs.append(f"({tup[0]}, {tup[1]}, len={tup[2]}m, max={tup[3]}, min={tup[4]}, sev={tup[5]})")
        # determine series-level severity for sorting: take max severity among intervals
        series_severity = float(g['severity_ratio'].replace([np.inf,-np.inf], np.nan).dropna().max()) if g['severity_ratio'].notna().any() else np.nan
        summary_rows.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'number_of_fault_intervals': num_intervals,
            'top_intervals': ' || '.join(interval_strs),
            'series_severity': series_severity
        })
    metric_fault_summary = pd.DataFrame(summary_rows)
    # sort by series_severity desc (NaNs last), then number_of_fault_intervals desc
    metric_fault_summary = metric_fault_summary.sort_values(['series_severity','number_of_fault_intervals'], ascending=[False,False]).reset_index(drop=True)
    metric_fault_summary = metric_fault_summary[['cmdb_id','kpi_name','number_of_fault_intervals','top_intervals','series_severity']].head(20)
    candidate_components_count = metric_fault_summary['cmdb_id'].nunique()

# Store full selected intervals for follow-up
metric_selected_intervals_full = selected_intervals_df.copy()

# Final compact outputs
metric_fault_summary, candidate_components_count, metric_selected_intervals_full.shape

```
Out[3]:
```
Summary of metric-level fault-interval detection (strict criteria applied):

What I did
- Computed global p95/p05 from the full series (no time filtering).
- Filtered data to incident window 2021-03-07 08:00–08:30 (UTC+8).
- Identified consecutive anomalous minutes per (cmdb_id, kpi) and applied strict rules:
  - High-is-bad KPIs: require global_p95 > 0, max_in_window ≥ 1.5 * global_p95, and interval length ≥ 2 minutes.
  - Low-is-bad KPIs: require global_p05 > 0, min_in_window ≤ 0.5 * global_p05, and length ≥ 2 minutes.
  - log.error_count: keep consecutive blocks with value > 0.

Result (compact)
- Only one metric-level fault candidate passed the strict filters:
  1) Component: IG01
     - KPI: OSLinux-CPU_CPU_CPUWio
     - Number of fault intervals: 1
     - Interval: 2021-03-07 08:08:00 — 2021-03-07 08:09:00 (UTC+8)
     - Length: 2 minutes
     - Severity ratio: ~1.548 (peak ≈ 55% above global p95)

Counts / stored data
- Distinct candidate faulty components found: 1 (IG01).
- Full interval details stored in kernel variable metric_selected_intervals_full for follow-up.

Interpretation / recommended next steps
- A short 2-minute fault interval for IG01 CPU Wio exceeded the strict breach threshold (≥50% above p95). This is the highest-confidence metric-level anomaly under the strict rules.
- Investigate IG01 around 08:08–08:09: check system traces, logs, and surrounding metrics (CPU, IO, threads, GC) to find root cause and any correlated symptoms.
- Because the selection removed noisy zero-threshold cases, this IG01 finding is a higher-priority, actionable candidate.

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                kpi_name  number_of_fault_intervals                                      top_intervals  series_severity
0    IG01  OSLinux-CPU_CPU_CPUWio                          1  (2021-03-07 08:08:00 CST, 2021-03-07 08:09:00 ...         1.548346, 1, (1, 15))```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz
import re

# reuse existing vars if present
tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_df is loaded and timestamps are timezone-aware
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
if 'timestamp' in trace_df.columns:
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Ensure global thresholds per (cmdb_id, trace_name) exist (compute if not)
try:
    trace_thresholds_full
except NameError:
    trace_thresholds_full = trace_df.groupby(['cmdb_id','trace_name'])['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        global_p05=lambda x: x.quantile(0.05)
    ).reset_index()

# 2) Filter trace records to IG01 and incident window
start = pd.Timestamp('2021-03-07 08:00:00', tz=tz)
end   = pd.Timestamp('2021-03-07 08:30:00', tz=tz)

ig01_traces_window = trace_df[(trace_df['cmdb_id']=='IG01') & (trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()

# store full IG01-in-window table for follow-up (kernel variable)
# ig01_traces_window saved as requested

# 3) Merge thresholds into IG01 window records
ig01_traces_window = ig01_traces_window.merge(trace_thresholds_full, on=['cmdb_id','trace_name'], how='left')

# 4) Classify trace_name as high-is-bad (latency/resource) or low-is-bad (throughput)
high_keywords = ['duration','duration_p95','cpu','memory','heap','threads','load','wio','systime','usertime','gc_pause','oom','maxtime','maxthreads']
low_keywords = ['row_count','rows','throughput']

def classify_trace(name):
    ln = str(name).lower()
    for kw in low_keywords:
        if kw in ln:
            return 'low_is_bad'
    for kw in high_keywords:
        if kw in ln:
            return 'high_is_bad'
    # default to high-is-bad (latency/resource)
    return 'high_is_bad'

ig01_traces_window['kpi_type'] = ig01_traces_window['trace_name'].apply(classify_trace)

# 5) Identify anomalous rows within window:
# - high-is-bad: value >= global_p95 (require global_p95 not null)
# - low-is-bad: value <= global_p05 (require global_p05 not null)
def is_trace_anomaly(row):
    val = row['value']
    typ = row['kpi_type']
    p95 = row['global_p95']
    p05 = row['global_p05']
    if typ == 'high_is_bad':
        if pd.isna(p95):
            return False
        return val >= p95
    else:
        if pd.isna(p05):
            return False
        return val <= p05

ig01_traces_window['is_anomaly'] = ig01_traces_window.apply(is_trace_anomaly, axis=1)

# 6) Compute severity_ratio:
def compute_severity(row):
    val = row['value']
    typ = row['kpi_type']
    p95 = row['global_p95']
    p05 = row['global_p05']
    if typ == 'high_is_bad':
        if pd.isna(p95) or p95 == 0:
            return np.nan
        return float(val) / float(p95)
    else:
        if pd.isna(p05) or val == 0:
            return np.nan
        return float(p05) / float(val)

ig01_traces_window['severity_ratio'] = ig01_traces_window.apply(compute_severity, axis=1)

# 7) Prepare anomaly rows DataFrame and select top 50 by severity_ratio desc
ig01_anoms = ig01_traces_window[ig01_traces_window['is_anomaly']].copy()

# format timestamp string for display
if not ig01_anoms.empty:
    ig01_anoms['timestamp_str'] = ig01_anoms['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Select columns requested: timestamp, trace_name, value, global_p95, global_p05, severity_ratio
anomalous_rows = pd.DataFrame(columns=['timestamp','trace_name','value','global_p95','global_p05','severity_ratio'])
if not ig01_anoms.empty:
    anomalous_rows = ig01_anoms[['timestamp_str','trace_name','value','global_p95','global_p05','severity_ratio']].rename(columns={'timestamp_str':'timestamp'}).copy()
    anomalous_rows = anomalous_rows.sort_values('severity_ratio', ascending=False).head(50).reset_index(drop=True)

# 8) Compact summary: total anomalous points, earliest anomalous timestamp, unique downstream targets in trace_name
total_anomalous_points = int(ig01_anoms.shape[0])
if total_anomalous_points > 0:
    earliest_ts = ig01_anoms['timestamp'].min().strftime('%Y-%m-%d %H:%M:%S %Z')
    # extract downstream targets like 'to_Tomcat01' using regex
    def extract_targets(name):
        return re.findall(r'to_[A-Za-z0-9]+', str(name))
    targets_series = ig01_anoms['trace_name'].apply(extract_targets)
    # flatten and unique
    unique_targets = sorted({t for lst in targets_series for t in lst})
else:
    earliest_ts = None
    unique_targets = []

# Prepare small summary DataFrame for display
summary_df = pd.DataFrame([{
    'component':'IG01',
    'total_anomalous_points': total_anomalous_points,
    'earliest_anomalous_timestamp': earliest_ts,
    'unique_downstream_targets': ', '.join(unique_targets) if unique_targets else ''
}])

# Final compact outputs: anomalous_rows (limit 50) and summary_df
anomalous_rows, summary_df

```
Out[4]:
```
Summary of IG01 trace anomalies in the incident window (2021-03-07 08:00—08:30 UTC+8). Global p95/p05 were computed from the full trace series before window filtering.

Key numbers
- Total IG01 trace-anomalous points in-window: 31
- Earliest anomalous timestamp: 2021-03-07 08:00:00 (UTC+8)
- Unique downstream targets referenced in trace names: to_Tomcat01, to_Tomcat02, to_Tomcat03, to_Tomcat04

Top anomalous rows (highest severity_ratio first, up to shown results)
1. 2021-03-07 08:15:00 — trace.to_Tomcat03.duration_mean: value=1.637902, global_p95=0.798152, severity≈2.05
2. 2021-03-07 08:15:00 — trace.to_Tomcat03.duration_p95: value=3.659000, global_p95=2.382150, severity≈1.54
3. 2021-03-07 08:23:00 — trace.to_Tomcat02.duration_mean: value=0.563188, global_p95=0.384953, severity≈1.46
4. 2021-03-07 08:15:00 — trace.self.duration_p95: value=2.778000, global_p95=2.066835, severity≈1.34
5. 2021-03-07 08:24:00 — trace.to_Tomcat04.duration_mean: value=0.499080, global_p95=0.379402, severity≈1.32
... (total 31 anomalous rows; list above shows the highest-severity examples)

Notes and interpretation
- The most significant anomalies are latency spikes toward Tomcat03 around 08:15 (duration_mean ~2× p95 and duration_p95 ~1.5× p95). These are the highest-confidence trace alerts in IG01.
- There are additional latency exceedances for routes to Tomcat02 and Tomcat04 and some elevated trace.self.duration_p95 values.
- Some throughput (row_count) points (e.g., to_Tomcat01 row_count = 36) meet the anomaly rule but show severity_ratio = 1.0 (equal to p05), so they are less severe than the latency breaches above.

Actionable next steps
- Prioritize investigating IG01 → Tomcat03 traffic around 08:15: check Tomcat03 metrics/logs (GC, thread, request queues) and network I/O/latency at that time.
- Correlate these trace latency spikes with metric-level IO/CPU anomalies (IG01 OSLinux CPU Wio was previously identified as a metric-level fault at ~08:08–08:09) to see if I/O issues preceded the trace latency.
- If you want, I can show the full list of the 31 anomalous IG01 trace rows or filter to a specific target (e.g., Tomcat03) for detailed timestamps and nearby values.

Note: The full IG01-in-window trace table is stored in kernel variable ig01_traces_window for follow-up.

The original code execution output of IPython Kernel is also provided below for reference:

(                  timestamp                       trace_name      value  global_p95  global_p05  severity_ratio
0   2021-03-07 08:15:00 CST  trace.to_Tomcat03.duration_mean   1.637902    0.798152    0.067074        2.052117
1   2021-03-07 08:15:00 CST   trace.to_Tomcat03.duration_p95   3.659000    2.382150    0.270927        1.536007
2   2021-03-07 08:23:00 CST  trace.to_Tomcat02.duration_mean   0.563188    0.384953    0.076715        1.463004
3   2021-03-07 08:15:00 CST          trace.self.duration_p95   2.778000    2.066835    0.499500        1.344084
4   2021-03-07 08:24:00 CST  trace.to_Tomcat04.duration_mean   0.499080    0.379402    0.073757        1.315440
..                      ...                              ...        ...         ...         ...             ...
26  2021-03-07 08:19:00 CST   trace.to_Tomcat01.duration_p95   1.167200    1.155140    0.536945        1.010440
27  2021-03-07 08:23:00 CST  trace.to_Tomcat01.duration_mean   0.312094    0.309493    0.095677        1.008404
28  2021-03-07 08:24:00 CST          trace.self.duration_p95   2.075700    2.066835    0.499500        1.004289
29  2021-03-07 08:02:00 CST      trace.to_Tomcat01.row_count  36.000000   98.100000   36.000000        1.000000
30  2021-03-07 08:01:00 CST      trace.to_Tomcat01.row_count  36.000000   98.100000   36.000000        1.000000

[31 rows x 6 columns],   component  total_anomalous_points earliest_anomalous_timestamp                          unique_downstream_targets
0      IG01                      31      2021-03-07 08:00:00 CST  to_Tomcat01, to_Tomcat02, to_Tomcat03, to_Tomc...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# timezone
tz = pytz.timezone('Asia/Shanghai')

# Reuse existing metric_df if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure timestamp is timezone-aware Asia/Shanghai
if metric_df['timestamp'].dt.tz is None:
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global p95 and p05 per (cmdb_id,kpi_name) from full series (do NOT filter first)
try:
    metric_thresholds_full
except NameError:
    metric_thresholds_full = metric_df.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95=lambda x: x.quantile(0.95),
        global_p05=lambda x: x.quantile(0.05)
    ).reset_index()

# 2) Filter metric.csv to IG01 and incident window (store full in kernel var ig01_metrics_window)
start = pd.Timestamp('2021-03-07 08:00:00', tz=tz)
end   = pd.Timestamp('2021-03-07 08:30:00', tz=tz)

ig01_metrics_window = metric_df[(metric_df['cmdb_id']=='IG01') & (metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# Merge thresholds
ig01_metrics_window = ig01_metrics_window.merge(metric_thresholds_full, on=['cmdb_id','kpi_name'], how='left')

# 3) Classify KPI types
high_keywords = ['duration','cpu','memory','heap','threads','load','wio','systime','usertime','gc_pause','oom','maxtime','maxthreads']
low_keywords = ['row_count','rows','throughput']

def classify_kpi(name):
    ln = str(name).lower()
    if ln == 'log.error_count':
        return 'error_count'
    for kw in low_keywords:
        if kw in ln:
            return 'low_is_bad'
    for kw in high_keywords:
        if kw in ln:
            return 'high_is_bad'
    return 'high_is_bad'  # default

ig01_metrics_window['kpi_type'] = ig01_metrics_window['kpi_name'].apply(classify_kpi)

# 4) Flag anomalies per rules
def flag_anomaly(row):
    val = row['value']
    typ = row['kpi_type']
    p95 = row['global_p95']
    p05 = row['global_p05']
    if typ == 'error_count':
        return bool(val > 0)
    if typ == 'low_is_bad':
        if pd.isna(p05):
            return False
        return val <= p05
    # high_is_bad
    if pd.isna(p95):
        return False
    return val >= p95

ig01_metrics_window['is_anomaly'] = ig01_metrics_window.apply(flag_anomaly, axis=1)

# 5) Compute severity_ratio
def compute_severity(row):
    val = row['value']
    typ = row['kpi_type']
    p95 = row['global_p95']
    p05 = row['global_p05']
    if typ == 'error_count':
        return float(val)
    if typ == 'low_is_bad':
        # severity = global_p05 / value (lower value => larger severity)
        if pd.isna(p05) or val == 0:
            return np.nan
        return float(p05) / float(val)
    # high_is_bad
    if pd.isna(p95) or p95 == 0:
        return np.nan
    return float(val) / float(p95)

ig01_metrics_window['severity_ratio'] = ig01_metrics_window.apply(compute_severity, axis=1)

# format timestamp string for display
ig01_metrics_window['timestamp_str'] = ig01_metrics_window['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# 6) Proximity note for two windows: 08:06–08:10 and 08:13–08:17 (inclusive)
prox1_start = pd.Timestamp('2021-03-07 08:06:00', tz=tz)
prox1_end   = pd.Timestamp('2021-03-07 08:10:00', tz=tz)
prox2_start = pd.Timestamp('2021-03-07 08:13:00', tz=tz)
prox2_end   = pd.Timestamp('2021-03-07 08:17:00', tz=tz)

def proximity_note(ts):
    notes = []
    if prox1_start <= ts <= prox1_end:
        notes.append('near_IOwait_08:06-08:10')
    if prox2_start <= ts <= prox2_end:
        notes.append('near_trace_spike_08:13-08:17')
    return ';'.join(notes) if notes else ''

ig01_metrics_window['proximity_note'] = ig01_metrics_window['timestamp'].apply(proximity_note)

# 7) Select anomalous rows and sort by severity_ratio desc, limit 50
ig01_anoms = ig01_metrics_window[ig01_metrics_window['is_anomaly']].copy()

# prepare display columns
if not ig01_anoms.empty:
    display_cols = ['timestamp_str','kpi_name','value','global_p95','global_p05','severity_ratio','proximity_note']
    anomalous_rows = ig01_anoms[display_cols].sort_values('severity_ratio', ascending=False).head(50).reset_index(drop=True)
    anomalous_rows = anomalous_rows.rename(columns={'timestamp_str':'timestamp'})
else:
    anomalous_rows = pd.DataFrame(columns=['timestamp','kpi_name','value','global_p95','global_p05','severity_ratio','proximity_note'])

# 8) Summary: total anomalies, earliest anomaly timestamp, unique KPIs anomalous, any anomaly at 08:15 exactly
total_anomalous_points = int(ig01_anoms.shape[0])
earliest_anomaly_ts = None
unique_kpis = []
anomaly_at_0815 = False
if total_anomalous_points > 0:
    earliest_anomaly_ts = ig01_anoms['timestamp'].min().strftime('%Y-%m-%d %H:%M:%S %Z')
    unique_kpis = sorted(ig01_anoms['kpi_name'].unique().tolist())
    # check exact 08:15:00
    exact_ts = pd.Timestamp('2021-03-07 08:15:00', tz=tz)
    anomaly_at_0815 = ig01_anoms['timestamp'].eq(exact_ts).any()

summary_df = pd.DataFrame([{
    'component':'IG01',
    'total_anomalous_points': total_anomalous_points,
    'earliest_anomalous_timestamp': earliest_anomaly_ts,
    'unique_anomalous_kpis': ', '.join(unique_kpis) if unique_kpis else '',
    'anomaly_at_08:15_exact': anomaly_at_0815
}])

# Final outputs: anomalous_rows (limit 50) and summary_df; full IG01-in-window metrics stored in ig01_metrics_window
anomalous_rows, summary_df

```
Out[5]:
```
Summary of IG01 metric correlations with the trace latency and IO-wait events (2021-03-07 08:00–08:30 UTC+8)

Quick facts
- Total IG01 metric anomalous points in-window: 670
- Earliest IG01 anomaly: 2021-03-07 08:00:00 (UTC+8)
- There is at least one IG01 metric anomaly exactly at 08:15:00 (True)
- Full IG01-in-window metric table stored in ig01_metrics_window for follow-up.

Top correlated findings (highest-severity examples)
- IO-wait event (~08:08–08:09)
  - OSLinux-CPU_CPU_CPUWio is strongly anomalous at:
    - 08:08:00 — value ≈ 0.5988, global_p95 ≈ 0.3867, severity ≈ 1.548 (proximate to IO-wait window)
    - 08:09:00 — value ≈ 0.5863, severity ≈ 1.516 (proximate to IO-wait window)
  - Also near that window:
    - OSLinux-CPU_CPU-2_SingleCpuUtil at 08:06:00 (severity ≈ 1.35, in 08:06–08:10 proximity)
  - These indicate increased CPU I/O wait and elevated CPU utilization around 08:06–08:10 consistent with the earlier detected IO-wait spike.

- Trace latency spikes (~08:15)
  - There is an anomaly exactly at 08:15 for IG01 (confirmed).
  - Metric anomalies in the 08:13–08:17 proximity window include:
    - OSLinux-CPU_CPU-0_SingleCpuidle at 08:13:00 (small severity >1.0, proximity to trace spike)
    - Other IG01 metrics have anomalies at 08:15 and nearby minutes (see ig01_metrics_window for full detail).
  - These support a temporal correlation between trace latency spikes around 08:15 and metric perturbations in CPU/idleness and other OS metrics.

- Disk / I/O metrics (other notable high-severity points)
  - OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps at 08:30:00 — severity ≈ 2.40 (large breach; later in the window)
  - OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps at 08:23:00 — severity ≈ 1.54
  - Several local-disk throughput/IO metrics show breaches later in the window (08:22–08:30), suggesting sustained or subsequent I/O activity or stress.

Interpretation and recommended next steps
- Strong evidence of an IO-related incident starting near 08:06–08:09 (CPU I/O wait spikes) — these are good candidate root contributors to downstream latency.
- Trace latency spikes at 08:15 (IG01 → Tomcat03 highest) temporally overlap with metric anomalies in the 08:13–08:17 window; investigate Tomcat targets for high request latency and correlate with IG01 I/O/Cpu states.
- Investigate storage subsystem and disk metrics (sda/sdb) because high disk Bps/Tps and IO wait coincide with or follow these spikes.
- Prioritize events with severity_ratio >> 1 (e.g., CPUWio >1.5× p95 and disk Bps >1.5× p95) — these are the highest-confidence correlations.
- Use ig01_metrics_window and the previously stored ig01_traces_window to examine the full timelines and correlate specific timestamps (start with 08:06–08:10 and 08:13–08:17, and exact 08:15).

If you want, I can:
- show the full top anomalous metric rows around 08:06–08:10 and 08:13–08:17,
- or produce a compact timeline of metric+trace anomalies minute-by-minute for IG01.

The original code execution output of IPython Kernel is also provided below for reference:

(                  timestamp                                           kpi_name         value    global_p95    global_p05  severity_ratio                proximity_note
0   2021-03-07 08:30:00 CST     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps  9.000000e+00  3.750000e+00  0.000000e+00        2.400000                              
1   2021-03-07 08:08:00 CST                             OSLinux-CPU_CPU_CPUWio  5.987530e-01  3.867050e-01  1.247150e-02        1.548346       near_IOwait_08:06-08:10
2   2021-03-07 08:23:00 CST     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  2.000000e+00  1.300000e+00  0.000000e+00        1.538462                              
3   2021-03-07 08:09:00 CST                             OSLinux-CPU_CPU_CPUWio  5.863000e-01  3.867050e-01  1.247150e-02        1.516143       near_IOwait_08:06-08:10
4   2021-03-07 08:06:00 CST                    OSLinux-CPU_CPU-2_SingleCpuUtil  2.662200e+00  1.971500e+00  7.821000e-01        1.350342       near_IOwait_08:06-08:10
..                      ...                                                ...           ...           ...           ...             ...                           ...
45  2021-03-07 08:29:00 CST                 OSLinux-OSLinux_ZABBIX_Host_Uptime  6.275969e+06  6.275938e+06  6.274312e+06        1.000005                              
46  2021-03-07 08:13:00 CST                    OSLinux-CPU_CPU-0_SingleCpuidle  9.908516e+01  9.908506e+01  9.857426e+01        1.000001  near_trace_spike_08:13-08:17
47  2021-03-07 08:22:00 CST  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvg...  1.070000e+00  1.070000e+00  1.070000e+00        1.000000                              
48  2021-03-07 08:21:00 CST  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...  2.000000e-01  2.000000e-01  0.000000e+00        1.000000                              
49  2021-03-07 08:22:00 CST     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  1.000000e+00  1.000000e+00  0.000000e+00        1.000000                              

[50 rows x 7 columns],   component  total_anomalous_points earliest_anomalous_timestamp                              unique_anomalous_kpis  anomaly_at_08:15_exact
0      IG01                     670      2021-03-07 08:00:00 CST  JVM-Memory_7778_JVM_Memory_HeapMemoryMax, JVM-...                    True)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```